In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 8: Cross-band. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_crossband as X, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The four Cross-band statement types (rules in scripts/edop/dashboard/wo02_rules_crossband.py)
for s in X.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print("distance = |percentile - 50| over rows with a usable percentile; zero-aware variables and variables with no catalog zero fraction are left out")
print(f"XB-01 names {X.TOP_BAND} per band (needs {X.MIN_BAND}+ eligible variables); XB-02 names {X.TOP_ALL} (needs {X.MIN_ALL}+); tail width for XB-03: {X.TAIL}")


XB-01  Furthest from the global median, within a band
      applies to: single basin, buffer, polity; Bands A to D (Band E has too few eligible variables); L6, L8
      template  : {band}: the variables furthest from the global median are {name} ({ord} percentile) and {name} ({ord}).
XB-02  Furthest from the global median, across all bands
      applies to: single basin, buffer, polity; L6, L8
      template  : Across all bands, the variables furthest from the global median are {name} ({ord}), {name} ({ord}) and {name} ({ord}).
XB-03  How many variables lie in the lowest or highest tenth
      applies to: single basin, buffer, polity; L6, L8
      template  : {k} of the {m} variables with a global percentile are at or below the 10th percentile or at or above the 90th.
XB-04  How far the place is from the global median overall
      applies to: single basin, buffer, polity; L6, L8
      template  : Across the {m} variables with a global percentile, the median distance from the global me

In [3]:
# Cell 3 — Fixtures: apply every rule; count where each applies and why not
fdf = A.apply_fixtures(fx, X)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of items):')
print(A.reason_table(fdf).to_string())


res                 applied  not_applicable
id    scope                                
XB-01 buffer             14              11
      polity             16               4
      single_basin       92              23
XB-02 buffer              5               0
      polity              4               0
      single_basin       23               0
XB-03 buffer              5               0
      polity              4               0
      single_basin       23               0
XB-04 buffer              5               0
      polity              4               0
      single_basin       23               0

why a statement did not apply (count of items):
id     reason      
XB-01  fewer_than_3    38


In [4]:
# Cell 4 — Draft sentences on chosen fixtures, to read and judge
for qid in ['F01', 'F04', 'F22', 'F32']:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  XB-01  Band A (physiographic): the variables furthest from the global median are Stream gradient (2nd percentile) and Slope (3rd percentile).
  XB-01  Band B (hydro-climatic): the variables furthest from the global median are Sand content (91st percentile) and Discharge monthly minimum (89th percentile).
  XB-01  Band C (bioclimatic): the variables furthest from the global median are Temperature annual mean (98th percentile) and Temperature monthly maximum (95th percentile).
  XB-01  Band D (anthropocene markers): the variables furthest from the global median are Human footprint 2009 (upstream) (72nd percentile) and Population density (71st percentile).
  XB-02  Across all bands, the variables furthest from the global median are Temperature annual mean (98th), Stream gradient (2nd) and Slope (3rd).
  XB-03  6 of the 28 variables with a global percentile are at or below the 10th percentile or at or above the 90th.
  XB-04  

In [5]:
# Cell 5 — Population sample: how often each statement applies (share of its own items), weighted and unweighted
pdf = A.apply_population(pop, W, X)
print('WEIGHTED by climate-zone size (percent of items; XB-01 items are bands, the others are queries):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of items):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3')


WEIGHTED by climate-zone size (percent of items; XB-01 items are bands, the others are queries):
kind   buffer L6  single L6  single L8
id                                    
XB-01       68.4       79.7       79.7
XB-02       99.0      100.0      100.0
XB-03       99.0      100.0      100.0
XB-04       99.0      100.0      100.0

UNWEIGHTED (percent of items):
kind   buffer L6  single L6  single L8
id                                    
XB-01       65.9       76.1       76.0
XB-02       99.0      100.0      100.0
XB-03       99.0      100.0      100.0
XB-04       99.0      100.0      100.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3


In [6]:
# Cell 6 — The numbers inside the statements (weighted)
print('XB-04 median distance from the global median (percentile points); XB-02 distance of the furthest variable; XB-03 share of variables in the tails:')
print(A.stat_quantiles(pdf, ['XB-04', 'XB-02', 'XB-03']).round(2).to_string(index=False))
print()
print('which variable is furthest from the global median most often (the first name in XB-02), weighted share of queries, by kind:')
for kind, label in [('s', 'single L6'), ('t', 'single L8'), ('b', 'buffer L6')]:
    lf = X.leader_frequency([q for q in pop if q['kind'] == kind], W)
    print(f"  {label}:", {k: round(v, 3) for k, v in list(lf.items())[:6]})
b1 = pdf[(pdf.id == 'XB-01') & pdf.applies]
print()
print('XB-01: weighted share of queries where each band gets a sentence (percent), single L6:')
g = b1[b1.kind == 'single L6'].groupby('variable').w.sum() / pdf[(pdf.id == 'XB-01') & (pdf.kind == 'single L6') & (pdf.variable == 'band A')].w.sum()
print((g * 100).round(1).to_string())


XB-04 median distance from the global median (percentile points); XB-02 distance of the furthest variable; XB-03 share of variables in the tails:
   id      kind   n   p10   p25   p50   p75   p90
XB-02 buffer L6  98 39.51 43.37 46.69 49.08 49.99
XB-02 single L6 306 40.06 45.35 48.57 50.00 50.00
XB-02 single L8 100 43.05 44.83 49.52 50.00 50.00
XB-03 buffer L6  98  0.00  0.10  0.17  0.24  0.34
XB-03 single L6 306  0.04  0.08  0.18  0.29  0.40
XB-03 single L8 100  0.03  0.08  0.17  0.25  0.38
XB-04 buffer L6  98 15.47 19.58 23.76 28.05 33.19
XB-04 single L6 306 15.94 20.10 24.67 30.38 35.85
XB-04 single L8 100 17.66 20.65 24.71 29.10 31.42

which variable is furthest from the global median most often (the first name in XB-02), weighted share of queries, by kind:
  single L6: {'Discharge monthly minimum': 0.092, 'Discharge annual mean': 0.082, 'Human footprint 2009': 0.074, 'Clay content (upstream)': 0.064, 'Temperature monthly minimum': 0.06, 'Silt content': 0.056}
  single L8: {'Dischar

In [7]:
# Cell 7 — Sweep: the tail width in XB-03 (10 means the lowest and highest tenth). Report a grid; the sentence states a count, not a label
import numpy as np
print('median share of variables in the lowest or highest `t` percentile points, by kind (a place of independent uniform percentiles would give 2t%):')
for kind, label in [('s', 'single L6'), ('t', 'single L8'), ('b', 'buffer L6')]:
    ts = X.tail_share([q for q in pop if q['kind'] == kind])
    wts = np.array([W[qid] for qid, _ in ts])
    row = {}
    for t in (5, 10, 15, 20, 25):
        v = np.array([d[t] for _, d in ts])
        row[f'{t}'] = round(P.wquantile(v, wts, 0.5), 3)
    print(f'  {label}: n {len(ts)}', row, '| uniform expectation', {str(t): round(2 * t / 100, 2) for t in (5, 10, 15, 20, 25)})
print()
print('share of queries with at least one variable in the lowest or highest tenth, weighted:')
for kind, label in [('s', 'single L6'), ('t', 'single L8'), ('b', 'buffer L6')]:
    ts = X.tail_share([q for q in pop if q['kind'] == kind], tails=(10,))
    wts = np.array([W[qid] for qid, _ in ts]); v = np.array([d[10] > 0 for _, d in ts])
    print(f'  {label}: {float((wts * v).sum() / wts.sum()):.3f}')


median share of variables in the lowest or highest `t` percentile points, by kind (a place of independent uniform percentiles would give 2t%):
  single L6: n 306 {'5': 0.107, '10': 0.179, '15': 0.286, '20': 0.393, '25': 0.5} | uniform expectation {'5': 0.1, '10': 0.2, '15': 0.3, '20': 0.4, '25': 0.5}
  single L8: n 100 {'5': 0.103, '10': 0.172, '15': 0.31, '20': 0.414, '25': 0.483} | uniform expectation {'5': 0.1, '10': 0.2, '15': 0.3, '20': 0.4, '25': 0.5}
  buffer L6: n 98 {'5': 0.067, '10': 0.174, '15': 0.265, '20': 0.369, '25': 0.459} | uniform expectation {'5': 0.1, '10': 0.2, '15': 0.3, '20': 0.4, '25': 0.5}

share of queries with at least one variable in the lowest or highest tenth, weighted:
  single L6: 0.917
  single L8: 0.967
  buffer L6: 0.880


In [8]:
# Cell 8 — Checks: which rows are eligible, the sanity of the world sample, and two catalog gaps
c = X.counts(pop + fx)
print('rows with a percentile, by band: eligible versus excluded (fixtures and population):')
for k, v in sorted(c.items()):
    print('  ', k, v)
print()
print('variables whose zero-aware status differs between L6 and L8:', X.zero_aware_level_difference())
print('catalog has no zero fraction for: reservoir_vol (see below)')
rv = [(q['kind'], r['representative_score']) for q in pop if q['kind'] == 's' for ctx, r in P.ae_rows(q) if r['variable'] == 'reservoir_vol' and r['status'] == 'ok']
wts = np.array([W[q['id']] for q in pop if q['kind'] == 's' for ctx, r in P.ae_rows(q) if r['variable'] == 'reservoir_vol' and r['status'] == 'ok'])
z = np.array([s == 0 for _, s in rv])
print(f'reservoir_vol, single L6: weighted share of basins with a percentile of exactly 0: {float((wts * z).sum() / wts.sum()):.3f} ({int(z.sum())} of {len(z)} sample rows)')
print()
q4 = A.stat_quantiles(pdf, ['XB-04'])
print('sanity: the median distance from the global median for a place with uniform percentiles is 25; the weighted sample gives:')
print(q4[['kind', 'n', 'p50']].round(1).to_string(index=False))


rows with a percentile, by band: eligible versus excluded (fixtures and population):
   ('A', 'eligible') 2271
   ('A', 'zero-aware (excluded)') 307
   ('B', 'eligible') 5717
   ('B', 'zero-aware (excluded)') 731
   ('C', 'eligible') 4125
   ('C', 'zero-aware (excluded)') 179
   ('D', 'eligible') 1552
   ('D', 'no zero fraction (excluded)') 507
   ('D', 'zero-aware (excluded)') 1043
   ('E', 'eligible') 104
   ('E', 'zero-aware (excluded)') 254

variables whose zero-aware status differs between L6 and L8: [('dist_sink', 0.3335, 0.1369)]
catalog has no zero fraction for: reservoir_vol (see below)
reservoir_vol, single L6: weighted share of basins with a percentile of exactly 0: 0.772 (252 of 306 sample rows)

sanity: the median distance from the global median for a place with uniform percentiles is 25; the weighted sample gives:
     kind   n  p50
buffer L6  98 23.8
single L6 306 24.7
single L8 100 24.7


In [9]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Cross-band rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
rows = []
for s in X.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    unit = 'bands' if sid == 'XB-01' else 'queries'
    row = dict(s, family='Cross-band')
    row['fixture_result'] = f"{int(f.applied.sum())} {unit} applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = (f'share of {unit}, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns) + '; polities not sampled')
    row['sweep_result'] = ('tail width swept 5 to 25 points (Cell 7); the sentence states a count' if sid == 'XB-03' else 'no cutoff')
    rows.append(row)
n_rows = P.upsert_master('Cross-band', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 44 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
